In [ ]:
import json
import string
import re
import sys
import pandas as pd
from collections import Counter
from bert_score import score as bert_score

#################################################################
# Part 1: 평가 지표 계산 함수 정의
#################################################################

def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def f1_score(prediction, ground_truth):
    """F1 스코어를 계산하는 함수"""
    prediction_tokens = normalize_answer(prediction).split()
    ground_truth_tokens = normalize_answer(ground_truth).split()
    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1


def exact_match_score(prediction, ground_truth):
    """Exact Match 스코어를 계산하는 함수"""
    return (normalize_answer(prediction) == normalize_answer(ground_truth))


def evaluate(ground_truth_dict, predictions_dict):
    """주어진 데이터셋에 대해 EM, F1, BERTScore를 계산하는 메인 함수"""
    f1 = exact_match = total = 0
    pred_list, gt_list = [], []

    for qid, ground_truth in ground_truth_dict.items():
        total += 1
        if qid not in predictions_dict:
            # print(f'Unanswered question {qid} will receive score 0.', file=sys.stderr)
            continue
        
        prediction = predictions_dict[qid]
        exact_match += exact_match_score(prediction, ground_truth)
        f1 += f1_score(prediction, ground_truth)

        # BERTScore 계산을 위해 예측과 정답을 리스트에 추가
        pred_list.append(str(prediction))
        gt_list.append(str(ground_truth))
        
    # BERTScore 계산 (GPU가 있다면 자동으로 사용)
    if len(pred_list) > 0:
        P, R, F1_bert = bert_score(pred_list, gt_list, lang='en', rescale_with_baseline=True, verbose=False)
        bert_f1 = float(F1_bert.mean())
    else:
        bert_f1 = 0.0

    if total == 0:
        return {'exact_match': 0.0, 'f1': 0.0, 'bert_score': 0.0}

    exact_match = 100.0 * exact_match / total
    f1 = 100.0 * f1 / total
    
    return {'exact_match': exact_match, 'f1': f1, 'bert_score': bert_f1 * 100}


#################################################################
# Part 2: 평가 실행 및 결과 출력
#################################################################

# ⭐⭐⭐ 평가하고 싶은 실험 번호를 여기에 입력하세요 (1, 2, 또는 3) ⭐⭐⭐
experiment_num = 1
# ⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐

print(f"========== Evaluation for Experiment #{experiment_num} ==========\n")

# --- 1. 정답 파일 로드 ---
golden_answers_file = f'triviaqa_golden_answers_comparison_{experiment_num}.json'
with open(golden_answers_file, 'r', encoding='utf-8') as f:
    golden_answers = json.load(f)

# --- 2. 각 예측 파일에 대해 평가 실행 ---
models = ["qwen", "gpt4o-mini"]
prompts = ["st_prompt", "prompt1", "prompt2"]
all_results = []

for model in models:
    for prompt in prompts:
        prediction_file = f"{model}_{prompt}_eval_results_triviaqa_comparison_{experiment_num}.json"
        
        try:
            with open(prediction_file, 'r', encoding='utf-8') as f:
                predictions = json.load(f)
        except FileNotFoundError:
            print(f"파일을 찾을 수 없습니다: {prediction_file}")
            continue

        # 평가 실행
        scores = evaluate(golden_answers, predictions)
        
        # 결과 저장
        result_row = {
            'model': model,
            'prompt': prompt,
            'EM': f"{scores['exact_match']:.2f}",
            'F1': f"{scores['f1']:.2f}",
            'BERTScore': f"{scores['bert_score']:.2f}"
        }
        all_results.append(result_row)
        
        print(f"--- Model: {model}, Prompt: {prompt} ---")
        print(json.dumps(scores, indent=2))
        print("-" * 35)

# --- 3. 최종 결과를 표(DataFrame)로 정리하여 출력 ---
results_df = pd.DataFrame(all_results)
print("\n========== Final Summary Table ==========")
display(results_df)